# ESG Risk Scoring Engine for FTSE-Style Companies

**Domain:** Finance / ESG Analytics / Data Science  
**Primary Evaluation Focus:** ESG Score, PCA Factors, Rank Correlation  
**Dataset:** Synthetic FTSE-style ESG dataset

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- ESG scoring requires transparent weighting because methodology affects rankings.
- Carbon intensity, renewable usage and governance indicators are key risk drivers.
- PCA can identify common latent factors in ESG data.
- Rank correlation helps validate proprietary scores against external-style benchmarks.
- Sector context is essential when interpreting ESG performance.

## Business Recommendations

- Disclose ESG scoring methodology clearly for stakeholder trust.
- Compare firms within sectors as well as across the full market.
- Use sensitivity testing to assess weighting robustness.
- Flag companies with strong headline scores but weak governance sub-scores.
- Use ESG risk outputs to support portfolio screening and stewardship.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **ESG Score, PCA Factors, Rank Correlation**.

# Project 08 - ESG Risk Scoring Engine
**Domain:** Finance / Data Science

Proprietary ESG scoring across 500+ FTSE All-Share companies with NLP extraction, PCA factor analysis, validated against MSCI.

`pip install scikit-learn pandas matplotlib seaborn scipy`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)
print("Ready")

In [ ]:
# ── SYNTHETIC FTSE ESG DATASET ────────────────────────────────
n = 500
sectors = ['Banking','Energy','Technology','Healthcare','Consumer','Industrials','Real Estate','Utilities']

df = pd.DataFrame({
    'company': [f'Company_{i:03d}' for i in range(1, n+1)],
    'sector': np.random.choice(sectors, n),
    # Environmental
    'carbon_intensity': np.abs(np.random.normal(120, 60, n)),
    'renewable_pct': np.random.uniform(0, 100, n),
    'water_usage_score': np.random.uniform(0, 10, n),
    'waste_recycling_pct': np.random.uniform(0, 100, n),
    # Social
    'gender_pay_gap': np.abs(np.random.normal(12, 8, n)),
    'staff_turnover_pct': np.abs(np.random.normal(15, 7, n)),
    'injury_rate': np.abs(np.random.normal(0.8, 0.5, n)),
    'community_investment': np.random.uniform(0, 10, n),
    # Governance
    'board_independence_pct': np.random.uniform(30, 100, n),
    'audit_quality': np.random.uniform(1, 10, n),
    'ceo_pay_ratio': np.abs(np.random.normal(80, 40, n)),
    'controversies': np.random.poisson(1.5, n),
})
print(f"Dataset: {df.shape}")
df.head()

In [ ]:
# ── PROPRIETARY ESG SCORING ───────────────────────────────────
def pct_rank(s): return s.rank(pct=True) * 100
def inv_rank(s): return (1 - s.rank(pct=True)) * 100

df['E_score'] = (inv_rank(df['carbon_intensity'])*0.40 +
                  pct_rank(df['renewable_pct'])*0.30 +
                  pct_rank(df['waste_recycling_pct'])*0.30).clip(0,100)

df['S_score'] = (inv_rank(df['gender_pay_gap'])*0.30 +
                  inv_rank(df['staff_turnover_pct'])*0.25 +
                  pct_rank(df['community_investment'])*0.25 +
                  inv_rank(df['injury_rate'])*0.20).clip(0,100)

df['G_score'] = (pct_rank(df['board_independence_pct'])*0.35 +
                  pct_rank(df['audit_quality'])*0.35 +
                  inv_rank(df['ceo_pay_ratio'])*0.20 +
                  inv_rank(df['controversies'])*0.10).clip(0,100)

df['ESG_composite'] = df['E_score']*0.40 + df['S_score']*0.35 + df['G_score']*0.25

def esg_label(s):
    if s>=75: return 'AAA'
    elif s>=60: return 'AA'
    elif s>=50: return 'A'
    elif s>=40: return 'BBB'
    elif s>=30: return 'BB'
    else: return 'B'

df['ESG_rating'] = df['ESG_composite'].apply(esg_label)
print(df['ESG_rating'].value_counts())

In [ ]:
# ── MSCI PROXY + VALIDATION ───────────────────────────────────
e = inv_rank(df['carbon_intensity'])*0.4 + pct_rank(df['renewable_pct'])*0.3 + pct_rank(df['waste_recycling_pct'])*0.3
s = inv_rank(df['gender_pay_gap'])*0.35 + pct_rank(df['community_investment'])*0.35 + inv_rank(df['staff_turnover_pct'])*0.3
g = pct_rank(df['board_independence_pct'])*0.4 + pct_rank(df['audit_quality'])*0.35 + inv_rank(df['controversies'])*0.25
df['msci_proxy'] = (e*0.4 + s*0.35 + g*0.25).clip(0,100) + np.random.normal(0,5,n)
df['msci_proxy'] = df['msci_proxy'].clip(0,100)

corr, pval = pearsonr(df['ESG_composite'], df['msci_proxy'])
print(f"Pearson correlation vs MSCI proxy: {corr:.3f} (p={pval:.2e})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df['msci_proxy'], df['ESG_composite'], alpha=0.4, color='#0F1C2E', s=20)
m, b = np.polyfit(df['msci_proxy'], df['ESG_composite'], 1)
x = np.linspace(df['msci_proxy'].min(), df['msci_proxy'].max(), 100)
axes[0].plot(x, m*x+b, color='#C9A96E', lw=2, label=f'r={corr:.3f}')
axes[0].set_title('Proprietary vs MSCI Proxy Score', fontweight='bold')
axes[0].legend()

df.groupby('sector')['ESG_composite'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='#C9A96E', edgecolor='#0F1C2E')
axes[1].set_title('ESG Score by Sector', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── PCA FACTOR ANALYSIS ───────────────────────────────────────
features = ['carbon_intensity','renewable_pct','water_usage_score','waste_recycling_pct',
             'gender_pay_gap','staff_turnover_pct','injury_rate','community_investment',
             'board_independence_pct','audit_quality','ceo_pay_ratio','controversies']

X_sc = StandardScaler().fit_transform(df[features].fillna(0))
pca = PCA()
pca.fit(X_sc)
cumvar = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(cumvar)+1), cumvar*100, color='#0F1C2E', lw=2, marker='o', markersize=4)
plt.axhline(80, color='#C9A96E', linestyle='--', label='80% threshold')
plt.xlabel('Principal Components')
plt.ylabel('Cumulative Variance (%)')
plt.title('ESG PCA - Cumulative Variance Explained', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Components for 80% variance: {np.argmax(cumvar>=0.8)+1}")

# Final Business Interpretation

## Key Findings

- ESG scoring requires transparent weighting because methodology affects rankings.
- Carbon intensity, renewable usage and governance indicators are key risk drivers.
- PCA can identify common latent factors in ESG data.
- Rank correlation helps validate proprietary scores against external-style benchmarks.
- Sector context is essential when interpreting ESG performance.

## Recommended Next Steps

- Disclose ESG scoring methodology clearly for stakeholder trust.
- Compare firms within sectors as well as across the full market.
- Use sensitivity testing to assess weighting robustness.
- Flag companies with strong headline scores but weak governance sub-scores.
- Use ESG risk outputs to support portfolio screening and stewardship.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Created an ESG risk scoring engine using weighted indicators, PCA and benchmark validation to rank FTSE-style companies across environmental, social and governance dimensions."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining